# Unity Catalog Functions – MCP Tool Wrappers

This notebook registers the Cookidoo MCP tools as **Unity Catalog (UC) functions**.
UC functions can be used as tools by Databricks AI agents, Genie spaces,
and SQL queries – providing a governed, discoverable interface to the MCP server.

**Architecture:**
```
Databricks Agent / Genie / SQL
        │
        ▼
  UC Function (cookidoo_agent.main.search_recipes)
        │
        ▼
  MCP Server (Docker container, REST API)
        │
        ▼
  Cookidoo Platform API
```

**Prerequisites:**
- Unity Catalog `cookidoo_agent` and schema `main` created
- MCP server URL stored in secret scope `cookidoo-mcp`
- Cookidoo credentials in secret scope `cookidoo-secrets`

## 1. Configuration

In [ ]:
KEY_VAULT_SCOPE = "cookidoo-secrets"
MCP_SCOPE = "cookidoo-mcp"
UC_CATALOG = "cookidoo_agent"
UC_SCHEMA = "main"

# In Azure, this is the Container Apps internal FQDN (private, VNet-only).
# Stored by scripts/deploy_azure_infrastructure.py during deployment.
try:
    MCP_SERVER_URL = dbutils.secrets.get(scope=MCP_SCOPE, key="mcp-server-url")
except Exception:
    # Fallback for local Docker testing only
    MCP_SERVER_URL = "http://localhost:8001"

print(f"Catalog:    {UC_CATALOG}")
print(f"Schema:     {UC_SCHEMA}")
print(f"MCP Server: {MCP_SERVER_URL}")

## 2. Create UC Functions via SQL

These SQL-based UC functions wrap HTTP calls to the MCP server.
Databricks AI agents automatically discover UC functions as tools.

In [ ]:
%%sql

USE CATALOG cookidoo_agent;
USE SCHEMA main;

In [ ]:
%%sql

-- UC Function: Search Recipes
-- Agents and Genie can call this function to search Cookidoo recipes.
CREATE OR REPLACE FUNCTION cookidoo_agent.main.search_recipes(
  query STRING COMMENT 'Natural language search query for recipes (e.g. "vegan pasta")',
  max_results INT DEFAULT 10 COMMENT 'Maximum number of results to return'
)
RETURNS STRING
COMMENT 'Search for recipes on the Cookidoo platform. Returns JSON array of recipe summaries with id, title, and description.'
LANGUAGE PYTHON
AS $$
  import requests
  import json

  mcp_url = "MCP_SERVER_URL_PLACEHOLDER"

  # Authenticate
  auth_resp = requests.post(
      f"{mcp_url}/auth/token",
      json={"username": "COOKIDOO_EMAIL_PLACEHOLDER", "password": "COOKIDOO_PASSWORD_PLACEHOLDER"},
      timeout=30,
  )
  token = auth_resp.json()["access_token"]

  # Search
  resp = requests.get(
      f"{mcp_url}/api/recipes/search",
      params={"q": query, "limit": max_results},
      headers={"Authorization": f"Bearer {token}"},
      timeout=30,
  )
  return json.dumps(resp.json().get("results", []))
$$;

In [ ]:
%%sql

-- UC Function: Get Recipe Details
CREATE OR REPLACE FUNCTION cookidoo_agent.main.get_recipe(
  recipe_id STRING COMMENT 'Unique Cookidoo recipe identifier'
)
RETURNS STRING
COMMENT 'Retrieve full details for a specific Cookidoo recipe including ingredients, steps, and nutritional information. Returns JSON.'
LANGUAGE PYTHON
AS $$
  import requests
  import json

  mcp_url = "MCP_SERVER_URL_PLACEHOLDER"

  auth_resp = requests.post(
      f"{mcp_url}/auth/token",
      json={"username": "COOKIDOO_EMAIL_PLACEHOLDER", "password": "COOKIDOO_PASSWORD_PLACEHOLDER"},
      timeout=30,
  )
  token = auth_resp.json()["access_token"]

  resp = requests.get(
      f"{mcp_url}/api/recipes/{recipe_id}",
      headers={"Authorization": f"Bearer {token}"},
      timeout=30,
  )
  return json.dumps(resp.json())
$$;

In [ ]:
%%sql

-- UC Function: Save Recipe
CREATE OR REPLACE FUNCTION cookidoo_agent.main.save_recipe(
  recipe_json STRING COMMENT 'JSON string of recipe data to save (must include title, ingredients, steps)'
)
RETURNS STRING
COMMENT 'Save a new or modified recipe to Cookidoo My Recipes collection. Input is a JSON string. Returns confirmation JSON.'
LANGUAGE PYTHON
AS $$
  import requests
  import json

  mcp_url = "MCP_SERVER_URL_PLACEHOLDER"
  recipe_data = json.loads(recipe_json)

  auth_resp = requests.post(
      f"{mcp_url}/auth/token",
      json={"username": "COOKIDOO_EMAIL_PLACEHOLDER", "password": "COOKIDOO_PASSWORD_PLACEHOLDER"},
      timeout=30,
  )
  token = auth_resp.json()["access_token"]

  resp = requests.post(
      f"{mcp_url}/api/recipes/save",
      json={"recipe": recipe_data, "collection": "My Recipes"},
      headers={"Authorization": f"Bearer {token}"},
      timeout=30,
  )
  return json.dumps(resp.json())
$$;

## 3. Register UC Functions as Agent Tools

With UC functions created, Databricks AI agents automatically discover
them. Here's how to explicitly configure an agent to use them.

In [ ]:
def create_uc_agent():
    """
    Create a Databricks AI agent that uses UC functions as tools.

    UC functions are automatically discovered by function name.
    """
    try:
        from databricks.agents import Agent, AgentConfig
        from databricks.agents.tools import UCFunctionTool

        tools = [
            UCFunctionTool(function_name="cookidoo_agent.main.search_recipes"),
            UCFunctionTool(function_name="cookidoo_agent.main.get_recipe"),
            UCFunctionTool(function_name="cookidoo_agent.main.save_recipe"),
        ]

        config = AgentConfig(
            model_serving_endpoint="cookidoo-agent-primary",
            tools=tools,
            instructions=(
                "You are a helpful cooking assistant with access to the Cookidoo "
                "recipe platform. Use the search_recipes tool to find recipes, "
                "get_recipe to see full details, and save_recipe to save "
                "user modifications. Be specific about ingredients and quantities."
            ),
        )

        agent = Agent(config=config)
        print("✓ Agent created with UC function tools:")
        for t in tools:
            print(f"  - {t.function_name}")
        return agent

    except ImportError:
        print("⚠ databricks.agents not available")
        print("  Use Databricks Runtime ML 14.3+ or install databricks-agents")
        return None


agent = create_uc_agent()

## 4. Test UC Functions

In [ ]:
%%sql

-- Test: Search for pasta recipes via UC function
SELECT cookidoo_agent.main.search_recipes('pasta', 3) AS results;

In [ ]:
%%sql

-- Test: Use in a query context
SELECT
  'pasta' AS query,
  cookidoo_agent.main.search_recipes('pasta', 5) AS results
UNION ALL
SELECT
  'risotto' AS query,
  cookidoo_agent.main.search_recipes('risotto', 5) AS results;

## 5. Genie Space Integration

These UC functions are automatically available in Genie spaces
that reference the `cookidoo_agent.main` schema. Users can ask:

- *"Search for vegan dinner recipes"*
- *"Show me the details of recipe XYZ"*
- *"Find gluten-free pasta recipes"*

Genie will call the UC functions and return formatted results.

## 6. Important Notes

### Security – Credential Handling

The UC function SQL definitions above use `PLACEHOLDER` values for
credentials. In production, replace with one of:

1. **Databricks Secrets** – Use `dbutils.secrets.get()` inside Python UDFs
   (requires a Python UC function, not SQL)
2. **Service Principal** – Configure the MCP server to accept
   service principal auth instead of username/password
3. **Pre-authenticated endpoint** – MCP server authenticated at startup
   via environment variables (recommended for Docker deployment)

### Cost Impact

- UC functions execute on the **calling cluster** (notebook or SQL warehouse)
- The MCP server runs in Docker – **no additional Databricks cost**
- Only cost is the HTTP roundtrip time added to queries